# Exploitation Zone — Spark batch (`spark_exploitation_zone.py`)

Same code as [`spark_exploitation_zone.py`](../spark_exploitation_zone.py). Spark reads metadata **directly from MinIO via S3A**:

- `s3a://<TRUSTED_ZONE_BUCKET>/metadata/observations.csv`
- `s3a://<EXPLOITATION_ZONE_BUCKET>/metadata/observations.csv` (anti-join)

Then: anti-join, `mapInPandas` on trusted `audio_path` for FFT/time-domain features, write `metadata/spark_pending_workset.json`.

**Docker CLI:** `docker compose run --rm exploitation-spark-batch`

## Paths and environment

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = None
for p in [_here, *_here.parents]:
    if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file():
        _root = p
        break
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
EXPLOITATION_ZONE_DIR = _root / "exploitation_zone"
TRUSTED_ZONE_DIR = _root / "trusted_zone"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if str(EXPLOITATION_ZONE_DIR) not in sys.path:
    sys.path.insert(1, str(EXPLOITATION_ZONE_DIR))
if str(TRUSTED_ZONE_DIR) not in sys.path:
    sys.path.insert(1, str(TRUSTED_ZONE_DIR))

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

os.environ["SPARK_ALLOW_HOST_DRIVER"] = "1"
os.environ["SPARK_MASTER"] = os.environ.get("SPARK_MASTER") or "spark://localhost:7077"
_s3a = (os.environ.get("SPARK_S3A_ENDPOINT") or "").strip()
if not _s3a or "host.docker.internal" in _s3a:
    os.environ["SPARK_S3A_ENDPOINT"] = "http://127.0.0.1:9000"

print("PROJECT_ROOT =", PROJECT_ROOT)

## Import batch logic

In [ ]:
import importlib

import spark_exploitation_zone

importlib.reload(spark_exploitation_zone)

from shared.minio_helpers import create_minio_client
from spark_exploitation_zone import run_spark_exploitation_zone

## Run: Spark features + pending workset

Same as `python exploitation_zone/spark_exploitation_zone.py` / `main()`.

In [ ]:
client = create_minio_client()
pending = run_spark_exploitation_zone(client)
print(f"Rows in spark_pending_workset: {len(pending)}")